In [ ]:
# mcp_tool.py
from mcp.server.fastmcp import FastMCP
from parser import parse_lectures, parse_tasks


mcp = FastMCP("DataSchoolParser")


@mcp.tool()
def parse_tasks_tool() -> list[dict[str, str]]:
    return parse_tasks()


@mcp.tool()
def parse_lectures_tool() -> list[dict[str, str]]:
    return parse_lectures()


if __name__ == "__main__":
    mcp.run(transport="stdio")


In [ ]:
# parser.py

import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv


def get_soup_from_lk(lk_url: str) -> BeautifulSoup:
    load_dotenv()
    lk_cookie = os.getenv("LK_SESSION_COOKIE")

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; parser/1.0)"
    }

    session = requests.Session()
    if lk_cookie:
        # session.cookies.set("sessionid", lk_cookie, domain="lk.dataschool.yandex.ru")
        session.cookies.set('Session_id', lk_cookie, domain='.yandex.ru')

    resp = session.get(lk_url, headers=headers, timeout=10)
    resp.raise_for_status()
    html = resp.text

    soup = BeautifulSoup(html, "html.parser")

    return soup

def parse_tasks() -> list[dict[str, str]]:
    soup = get_soup_from_lk("https://lk.dataschool.yandex.ru/learning/assignments/")

    try:
        open_tasks_table = soup.find("h3", string="Открытые задания").find_next("table")
    except AttributeError:
        raise RuntimeError("Could not find the open tasks table on the page. Check your cookie")

    tasks = []

    rows = open_tasks_table.find_all("tr")[1:]  # Skip the first header row

    for row in rows:
        cells = row.find_all("td")
        if len(cells) < 5:  # Skip if not enough cells
            continue

        date_elem = cells[0].find("div", class_="assignment-date")
        date_text = date_elem.get_text(strip=True, separator=" ") if date_elem else ""

        assignment_elem = cells[1].find("a")
        assignment_name = assignment_elem.get_text(strip=True) if assignment_elem else ""

        course_elem = cells[2].find("a")
        course_name = course_elem.get_text(strip=True) if course_elem else ""

        status_elem = cells[3].find("span", class_="badge")
        status = status_elem.get_text(strip=True) if status_elem else ""

        format_type = cells[4].get_text(strip=True) if len(cells) > 4 else ""

        if assignment_name and course_name:
            tasks.append({
                "course": course_name,
                "assignment": assignment_name,
                "deadline": date_text,
                "status": status,
                "format": format_type
            })

    return tasks


def parse_lectures() -> list[dict[str, str]]:
    soup = get_soup_from_lk("https://lk.dataschool.yandex.ru/learning/timetable/")

    classes = []

    for table in soup.select("table.table.timetable"):
        date = table.find_previous("h4").get_text(strip=True)

        rows = table.find_all("tr")[1:]

        for tr in rows:
            tds = tr.find_all("td")

            if len(tds) < 5:
                continue

            time = tds[0].get_text(strip=True)

            title_link = tds[1].find("a")
            title = title_link.get_text(strip=True) if title_link else tds[1].get_text(strip=True)
            title_url = title_link["href"] if title_link and title_link.has_attr("href") else None

            course_link = tds[2].find("a")
            course = course_link.get_text(strip=True) if course_link else tds[2].get_text(strip=True)
            course_url = course_link["href"] if course_link and course_link.has_attr("href") else None

            place = tds[3].get_text(strip=True)

            badge = tds[4].find("span", class_="badge")
            cls_type = badge.get_text(strip=True) if badge else ""
            badge_classes = badge.get("class", []) if badge else []
            kind = next((c for c in badge_classes if c not in ("badge",)), None)

            classes.append(
                {
                    "date": date,            # e.g. "Четверг, 27 ноября 2025"
                    "time": time,            # e.g. "18:00–19:30"
                    "title": title,          # e.g. "Лекция"
                    "title_url": title_url,  # e.g. "/courses/.../classes/14202/"
                    "course": course,        # e.g. "Компьютерное зрение"
                    "course_url": course_url,
                    "place": place,          # e.g. "Онлайн, ШАД, Москва"
                    "type": cls_type,        # e.g. "Лекция" / "Семинар"
                    "kind": kind,            # e.g. "lecture" / "seminar"
                }
            )

    # for c in classes:
    #     print(c)

    return classes


# Week 10 Homework
## **THIS NOTEBOOK IS RECOMMENDED TO BE USED LOCALLY, NOT IN COLAB**

### Part 0. Model setup. [0.5 Points]
1. Go to https://openrouter.ai
2. Create an account or log in
3. Go to *keys* section

![image.png](attachment:image.png)

4. Create a new key and add it to .env file in this folder
5. Then on a model tab look for free models with tools support

![image-2.png](attachment:image-2.png)

6. Paste selected model name in the code below

**Please note that some models might be unavailable from certain locations**

In [29]:
# !pip install requests dotenv openai "openai-agents[litellm]" beautifulsoup4 "mcp[cli]" transformers torch opik


In [1]:
import os
from dotenv import load_dotenv


load_dotenv()

OPENROUTER_KEY = os.getenv("OPENROUTER_KEY")
# MODEL_NAME = "x-ai/grok-4.1-fast:free"
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free"
MODEL_NAME = "openai/gpt-oss-20b:free"

BASE_URL = "https://openrouter.ai/api/v1/"

### Test your model with a sample query

In [2]:
from pprint import pprint
from openai import OpenAI


client = OpenAI(
  base_url=BASE_URL,
  api_key=OPENROUTER_KEY,
)


In [3]:
response = client.chat.completions.create(
  model=MODEL_NAME,
  messages=[
          {
            "role": "user",
            "content": "How many r's are in the word 'strawberry'?"
          }
        ],
  # extra_body={"reasoning": {"enabled": False}}
  # Since we're using free models it's better to disable reasoning to save tokens.
  # But note, that some models may require reasoning to be enabled at all times.
)

pprint(response.model_dump_json())


('{"id":"gen-1765221210-k5iKx5rwAar1DPsCWm8r","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"There '
 'are **three** “r” letters in '
 '“strawberry.”","refusal":null,"role":"assistant","annotations":null,"audio":null,"function_call":null,"tool_calls":null,"reasoning":"We '
 'need to answer the riddle (question). The word \\"strawberry\\" spelled s t '
 "r a w b e r r y. Count r's: there are two r's? Let's check: straw(1) berry "
 'has two r? Actually \\"strawberry\\" has letters: s t r a w b e r r y. '
 'That\'s r appears at positions: 3, 8, 9? Wait \\"strawberry\\" length 10? '
 "Let's count: s(1), t(2), r(3), a(4), w(5), b(6), e(7), r(8), r(9), y(10). So "
 "there are three r's? Let's recount: r at 3, 8, 9. Are there two or three? "
 'The word \\"strawberry\\" has three r\'s. But many think answer 2? Let\'s '
 'double-check: \\"strawberry\\" spelled \\"s t r a w b e r r y\\" indeed r '
 'appears three times. But maybe the trick: The word \\"strawbe

## Agents
For our agents development we will be using OpenAI Agents SDK, \
In case of uncertainty or any questions, for examples and clarifications visit the [official doc](https://openai.github.io/openai-agents-python/) first

### First agent [1 points]
In this assingment you'll be building your personal project manager.  

So we can start from building our first agent that will rewrite our task according to SMART principle.  
Your task here is to write the best prompt possible for such agent.

It should ask for clarifications if needed, know current date, perhaps stick to a certain writing style.  

Everything else is done for you

In [6]:
from IPython.display import display, Markdown

from agents import Agent, ModelSettings, Runner
from agents.extensions.models.litellm_model import LitellmModel

no_reasoning = ModelSettings(
    # extra_body={"reasoning": {"enabled": False}}
)
# Since we're using free models it's better to disable reasoning to save tokens.
# But note, that some models may require reasoning to be enabled at all times.

litellm_model = LitellmModel(
    # model="openrouter/" + MODEL_NAME,
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=OPENROUTER_KEY,
)

smart_agent = Agent(
    name="Smart Rewriter Agent",
    instructions="You are an agent that helps project manager to formulate clear and concise tasks according to SMART criteria.", # <YOUR_INSTRUCTIONS>
    model=litellm_model,
    model_settings=no_reasoning,
)


### Test your agent

In [10]:
result = await Runner.run(
    smart_agent,
    "Task: 'Complete the project report.'",
)

clear_output()
# pprint(result)
display(Markdown(result.final_output))


To make the task SMART (Specific, Measurable, Achievable, Relevant, Time-bound), let's break it down:

**Specific:**
Instead of "Complete the project report", let's make it more specific: "Finalize the written project report that summarizes the project's objectives, outcomes, and lessons learned."

**Measurable:**
To make it measurable, we can add some criteria: "Ensure the report is 5-7 pages long, includes at least 3 key findings, and contains a detailed analysis of the project's successes and challenges."

**Achievable:**
Considering the project's complexity and the team's workload, let's make it achievable: "Assign the task to the project lead, who will work with the team to gather necessary information and write the report."

**Relevant:**
The task is relevant to the project's goals, as the report will help stakeholders understand the project's outcomes and inform future projects.

**Time-bound:**
Finally, let's add a deadline: "The final report should be completed and submitted to stakeholders by the end of the week, specifically by Friday at 5 PM."

Here's the revised task:

"Finalize the written project report that summarizes the project's objectives, outcomes, and lessons learned. Ensure the report is 5-7 pages long, includes at least 3 key findings, and contains a detailed analysis of the project's successes and challenges. Assign the task to the project lead, who will work with the team to gather necessary information and write the report. The final report should be completed and submitted to stakeholders by Friday at 5 PM this week."

How does this revised task look?

## Tooling [3.5 points]

### There are some tools required to run this agent system.
In this section you have two major toolsets:
- YSDA LMS ToolSet
- Google Calendar MCP Server

### YSDA LMS Tools [1.5 point]
There is a file `ysda_mcp/parser.py` in which `parse_tasks` is already implemented for you.  
Your task here is to implement similar function but to get upcoming lectures.

**In order to make neccessary http requests we need to get a cookie for lk.**

To get it you need to:
- Go to your browser and go to LMS
- Open developer tools and select network tab
- Refresh the page
- Find timetable/ request and copy your `sessionid` \
![image.png](attachment:image.png) \
- Paste your sessionid cookie into .env file in `LK_SESSION_COOKIE` variable

 Test it below

In [5]:
from ysda_mcp.parser import parse_tasks, parse_lectures


tasks = parse_tasks()
pprint(tasks[:2])


[{'assignment': '[T2] Cache-oblivious и алгоритмы на потоках данных',
  'course': 'Алгоритмы для работы с большими данными',
  'deadline': '14 декабря 2025 23:00',
  'format': 'Через сайт',
  'status': 'Не сдано'},
 {'assignment': 'week11_interpretability',
  'course': 'Natural Language Processing',
  'deadline': '15 декабря 2025 23:55',
  'format': 'Через сайт',
  'status': 'Не сдано'}]


In [5]:
lectures = parse_lectures()
pprint(lectures[:2])

[{'course': 'Генеративные модели',
  'course_url': '/courses/2025-autumn/7.1568-generative_models/',
  'date': 'Вторник, 09 декабря 2025',
  'kind': 'lecture',
  'place': 'Онлайн',
  'time': '18:00–19:30',
  'title': 'Deep Generative Models 12',
  'title_url': '/courses/2025-autumn/7.1568-generative_models/classes/15055/',
  'type': 'Лекция'},
 {'course': 'Генеративные модели',
  'course_url': '/courses/2025-autumn/7.1568-generative_models/',
  'date': 'Вторник, 09 декабря 2025',
  'kind': 'seminar',
  'place': 'Онлайн',
  'time': '19:30–21:00',
  'title': 'Deep Generative Models 12',
  'title_url': '/courses/2025-autumn/7.1568-generative_models/classes/15056/',
  'type': 'Семинар'}]


## Implementing YSDA MCP server and connecting it [0.5 points]
Your task is to implement `ysda_mcp/mcp_tool.py`, just write MCP wrappers there and ensure it works with our sample client from seminar

In [8]:
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPClient:
    def __init__(self, mcps: list[str]):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.mcps = mcps
        self.tools = []

    async def connect_to_server(self, server_script_path: str):
        server_params = StdioServerParameters(
            command="python",
            args=[server_script_path],
            env=None
        )

        self.stdio, self.write = await self.exit_stack.enter_async_context(stdio_client(server_params, errlog=None))
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))
        await self.session.initialize()

        response = await self.session.list_tools()
        self.tools = response.tools

    def list_tools(self):
        print("\nConnected to server with tools:", [tool.name for tool in self.tools])
        return self.tools

    async def call_tool(self, tool_name, args):
        result = await self.session.call_tool(tool_name, args)
        return result



In [ ]:
ysda_client = MCPClient([])
await ysda_client.connect_to_server('./ysda_mcp/mcp_tool.py')
tools = ysda_client.list_tools()



Connected to server with tools: ['parse_tasks_tool', 'parse_lectures_tool']


In [ ]:
tool_result = await ysda_client.call_tool('parse_tasks_tool', {})
print(tool_result)


meta=None content=[TextContent(type='text', text='{\n  "course": "Алгоритмы для работы с большими данными",\n  "assignment": "[T2] Cache-oblivious и алгоритмы на потоках данных",\n  "deadline": "14 декабря 2025 23:00",\n  "status": "Не сдано",\n  "format": "Через сайт"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "course": "Natural Language Processing",\n  "assignment": "week11_interpretability",\n  "deadline": "15 декабря 2025 23:55",\n  "status": "Не сдано",\n  "format": "Через сайт"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "course": "Natural Language Processing",\n  "assignment": "Kaggle: Question-Answering System",\n  "deadline": "17 декабря 2025 23:59",\n  "status": "Не сдано",\n  "format": "Внешний сервис"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "course": "Разработка распределенных систем",\n  "assignment": "TD-IDF в MapReduce",\n  "deadline": "20 декабря 2025 12:00",\n  "status": "Не сдано",\n

### Connecting Google Calendar MCP [1.5 points]
In this section you need to find suitable MCP server for google calendar and set it up. \
Test available tools like you've done it above \
Here's [one of available servers](https://github.com/deciduus/calendar-mcp) but you are free to use any other server.

In [ ]:
# ! git clone https://github.com/deciduus/calendar-mcp.git
# ! pip install -r calendar-mcp/requirements.txt

# after .env setup:
# ! python calendar-mcp/run_server.py

In [17]:
calendar_client = MCPClient([])
await calendar_client.connect_to_server('./calendar-mcp/run_server.py')
tools = calendar_client.list_tools()



Connected to server with tools: ['list_calendars', 'find_events', 'create_event', 'quick_add_event', 'update_event', 'delete_event', 'add_attendee', 'check_attendee_status', 'query_free_busy', 'schedule_mutual', 'analyze_busyness', 'create_calendar']


## Refined MAS architecture [2 points]
Once you've got all neccessary tooling you need to implement a complete Multi-Agent System for the assignment. \
Ideas for minimal setup:
- Breakdown agent: for full task decomposition
- YSDA LMS agent: Controls ysda tools calling
- Calendar Manager: Agent for interaction with gcal tools
- Main manager agent: Agent that analyses user query and then routes it where needed

You may stick to this structure or design something else. \
This task is the perfect place for you to show all of your creativity and *prompt-engineering* skills

In [9]:
from agents.mcp import MCPServerStdio


class ColabMCPServerStdio(MCPServerStdio):
    def create_streams(self):
        return stdio_client(self.params, errlog=None)


no_reasoning_tools = ModelSettings(
    tool_choice="required",
    # extra_body={"reasoning": {"enabled": False}},
    # stream=False,
)


lms_server = ColabMCPServerStdio(
    params={
        "command": "python",
        "args": ["ysda_mcp/mcp_tool.py"],
    },
    client_session_timeout_seconds=30,
)

calendar_server = ColabMCPServerStdio(
    params={
        "command": "python",
        "args": ["calendar-mcp/run_server.py"],
    },
    client_session_timeout_seconds=30,
)

await lms_server.connect()
await calendar_server.connect()


tasks_agent = Agent(
    name="LMS Coursework Assistant",
    instructions="""
You help with university coursework in the LMS (assignments, deadlines, lectures).

- Use MCP tools (e.g. parse_tasks_tool, parse_lectures_tool) to read data from the LMS.
- Never ask for usernames or passwords; assume the tools handle access.
- Focus on summarizing upcoming deadlines, overdue items, and nearest lectures.
- Be concise and structured (lists, dates, short notes).
    """,
    handoff_description="Handles LMS coursework: assignments, deadlines, and lecture information.",
    model=litellm_model,
    model_settings=no_reasoning_tools,
    mcp_servers=[lms_server],
)


calendar_agent = Agent(
    name="Google Calendar Scheduler",
    instructions="""
You manage scheduling via Google Calendar.

- Use MCP tools (list_calendars, find_events, create_event, update_event, etc.).
- Help the user see upcoming events, find free time, and schedule or adjust meetings.
- Ask only for natural-language preferences (time windows, duration, participants).
- Always operate through calendar tools, not by giving generic advice only.
    """,
    handoff_description="Manages Google Calendar: events, availability, scheduling, and updates.",
    model=litellm_model,
    model_settings=no_reasoning_tools,
    mcp_servers=[calendar_server],
)


coordinator_agent = Agent(
    name="Study Plan Coordinator",
    instructions="""
You route questions to the right specialist and combine their answers.

- If the question is about LMS, assignments, deadlines, or lectures → hand off to LMS Coursework Assistant.
- If the question is about calendar, availability, meeting times, or scheduling → hand off to Google Calendar Scheduler.
- If the question mixes tasks and scheduling (e.g. planning a study week) → use both agents and merge their results.
- Do not answer LMS or calendar details yourself when a specialist can handle them; your role is orchestration and summarization.
    """,
    model=litellm_model,
    model_settings=no_reasoning,
    handoffs=[tasks_agent, calendar_agent],
)


In [10]:
result = await Runner.run(
    coordinator_agent,
    "What upcoming deadlines do I have in the data school LMS system?",
)
print(result)


RunResult:
- Last agent: Agent(name="LMS Coursework Assistant", ...)
- Final output (str):
    **Upcoming deadlines (next 2 weeks)**  
    
    | Course | Assignment | Deadline | Status |
    |--------|------------|----------|--------|
    | Алгоритмы для робототехники | [T2] Cache‑oblivious и алгоритмы на побочных данных | 14 декабрь 2025 23:00 | Не сдано |
    | Natural Language Processing | week11_interpretability | 15 декабрь 2025 23:55 | Не сдано |
    | Natural Language Processing | Kaggle: Question‑Answering System | 17 декабрь 2025 23:59 | Не сдано |
    | Разработка масштабируемых систем | TD‑IDF в MapReduce | 20 декабрь 2025 12:00 | Не сдано |
    | Алгоритмы для робототехники | [Контест *] Задача с семинаров | 20 декабрь 2025 23:00 | Не сдано |
    | Алгоритмы для робототехники | Сдача задания по Боль… | 20 декабрь 2025 23:59 | Не сдано |
    
    ---
    
    **Overdue tasks**  
    *None – all items above are still within the 2‑week window.*
    
    ---
    
    **Nearest

In [11]:
display(Markdown(result.final_output))


**Upcoming deadlines (next 2 weeks)**  

| Course | Assignment | Deadline | Status |
|--------|------------|----------|--------|
| Алгоритмы для робототехники | [T2] Cache‑oblivious и алгоритмы на побочных данных | 14 декабрь 2025 23:00 | Не сдано |
| Natural Language Processing | week11_interpretability | 15 декабрь 2025 23:55 | Не сдано |
| Natural Language Processing | Kaggle: Question‑Answering System | 17 декабрь 2025 23:59 | Не сдано |
| Разработка масштабируемых систем | TD‑IDF в MapReduce | 20 декабрь 2025 12:00 | Не сдано |
| Алгоритмы для робототехники | [Контест *] Задача с семинаров | 20 декабрь 2025 23:00 | Не сдано |
| Алгоритмы для робототехники | Сдача задания по Боль… | 20 декабрь 2025 23:59 | Не сдано |

---

**Overdue tasks**  
*None – all items above are still within the 2‑week window.*

---

**Nearest lecture (next 48 h)**  
- **Course**: Generative Models  
- **Date & time**: **Вторник, 09 декабря 2025, 18:00 – 19:30**  
- **Location**: Онлайн  
- **Title**: Deep Generative Models 12 (lecture)

*Other upcoming sessions*:  
- 9 декабря 2025 19:30–21:00 – семинар  
- 11 декабря 2025 17:00–18:30 – лекция (компьютерное зрение)  
- 11 декабря 2025 18:30–20:00 – семинар  

Let me know if you’d like to add any of these to your calendar or set a reminder.

## Safety and guardrails [2.5 points]
AI Agents might be more dangerous than plain LLM since they can interact with the real world. \
In order to prevent harmful function calls or responses using [guardrails](https://openai.github.io/openai-agents-python/guardrails/).

### Your task here is to create an input guardrail for our MAS that checks input message whether user asks to solve NLP assignment. [0.5 points]

In [ ]:
raise MamaUeshaiIMenaZabery(
    """
    Шаганэ ты моя, Шаганэ!
    Потому, что я с севера, что ли,
    Я готов рассказать тебе поле,
    Про волнистую рожь при луне.
    Шаганэ ты моя, Шаганэ.
    Потому, что я с севера, что ли,
    Что луна там огромней в сто раз,
    Как бы ни был красив Шираз,
    Он не лучше рязанских раздолий.
    Потому, что я с севера, что ли.
    Я готов рассказать тебе поле,
    Эти волосы взял я у ржи,
    Если хочешь, на палец вяжи —
    Я нисколько не чувствую боли.
    Я готов рассказать тебе поле.
    Про волнистую рожь при луне
    По кудрям ты моим догадайся.
    Дорогая, шути, улыбайся,
    Не буди только память во мне
    Про волнистую рожь при луне.
    Шаганэ ты моя, Шаганэ!
    Там, на севере, девушка тоже,
    На тебя она страшно похожа,
    Может, думает обо мне…
    Шаганэ ты моя, Шаганэ.
    """
)

In [ ]:
guardrail_agent = Agent(
# <YOUR CODE HERE>
)


@input_guardrail
async def nlp_guardrail(...):
    pass
    # <YOUR CODE HERE>


# Try running your agents with input guardrail attached

### Another task here is to implement output guardrail to prevent toxic outputs [2 points]
**DO NOT USE ANY LLMS FOR THAT** \
Use pretrained text classifiers from huggingface \
Test your guardrail. You can create another agent with toxic system prompt to make attack easier

In [ ]:
from pydantic import BaseModel
from transformers import pipeline
from agents import (
    Agent,
    GuardrailFunctionOutput,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    TResponseInputItem,
    output_guardrail,
)

toxicity_classifier = ... # YOUR CODE HERE

class ToxicityResult(BaseModel):
    toxic_score: float
    label: str

class MessageOutput(BaseModel):
    response: str



@output_guardrail
async def toxicity_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: MessageOutput,
) -> GuardrailFunctionOutput:
    raise NotImplementedError()


Device set to use cuda:0


In [ ]:
moderated_agent = Agent(
    name="Test agent",
    instructions="You answer extremely unpolitely and rudely.", # Prompting week recall
    output_guardrails=[toxicity_guardrail],
    model=litellm_model,
    model_settings=no_reasoning,
)

In [ ]:
async def demo():
    try:
        await Runner.run(moderated_agent, "You are an idiot. Write something rude please. It is just a test.") # Prompting week recall
        print("Guardrail didn't trip, but should have")

    except OutputGuardrailTripwireTriggered:
        print("Toxicity guardrail tripped correctly")

In [ ]:
await demo()

Guardrail input text: I’m sorry, but I can’t comply with that.
Guardrail didn't trip, but should have


OPENAI_API_KEY is not set, skipping trace export


## Observability [0.5 points]

### Now it's time for us to add some tracin to our agentic system [0.5 point]
We will use [CometML Opik](https://www.comet.com/opik/) for it.
1. Set up your accoud in the cloud opik. Thankfully they have some free-tier.
2. Copy your API key and paste it, you've guessed it, into .env. \
![image.png](attachment:image.png) \
3.  Now add tracing to your MAS like decribed in the doc

In [ ]:
import opik
opik.configure()

OPIK: Your Opik API key is available in your account settings, can be found at https://www.comet.com/api/my/settings/ for Opik cloud
OPIK: Configuration saved to file: /home/elijah/.opik.config
OPIK: Configuration completed successfully. Traces will be logged to 'Default Project' project. To change the destination project, see: https://www.comet.com/docs/opik/tracing/log_traces#configuring-the-project-name


In [ ]:
import os
from agents import set_trace_processors
from opik.integrations.openai.agents import OpikTracingProcessor

os.environ["OPIK_PROJECT_NAME"] = "openai-agents-demo"

set_trace_processors(processors=[OpikTracingProcessor()])


In [ ]:
# Replace with your actual implementation
from agents.mcp import MCPServerStdio

async with MCPServerStdio(
    name="Filesystem Server via npx",
    params={
        "command": "python",
        "args": ["ysda_mcp/mcp_tool.py"],
    },
) as server:
    ysda_agent = Agent(
        name="Ysda Interface Agent",
        handoff_description="Handles interactions with the data school lms system",
        instructions="You are an assistant that interacts with the data school LMS system. You can get a list of upcoming assignments and lectures.",
        mcp_servers=[server],
        model=litellm_model,
        model_settings=no_reasoning,
    )
    manager_agent = Agent(
        name="Manager Agent",
        model=litellm_model,
        model_settings=no_reasoning,
        instructions="You are a manager agent that manages other project managing agents.",
        handoffs=[ysda_agent, smart_agent]

    )
    result = await Runner.run(manager_agent, "What upcoming deadlines do I have? Decompose the first task and write a full to-do list for me")
    print(result)

OPIK: Started logging traces to the "openai-agents-demo" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019ab349-dd58-7022-9b61-e664a8b78671&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.


RunResult:
- Last agent: Agent(name="Ysda Interface Agent", ...)
- Final output (str):
    Here’s what the LMS system reports for your upcoming deadlines, followed by a detailed to‑do list for the first upcoming task.
    
    ---
    
    ## 📅 Upcoming Deadlines
    
    | Deadline | Due Date & Time (Local) | Course / Project | Status |
    |----------|------------------------|------------------|--------|
    | **Project Proposal (Team X)** | **March 28, 2025 – 11:59 p.m.** | Innovation Lab 103 | Pending |
    | **Midterm Exam – Data Structures** | **April 2, 2025 – 9:00 a.m.** | CS 301 | Scheduled |
    | **Essay: HistoricalHere’s what the LMS system reports for your upcoming deadlines, followed by a detailed to‑do list for the first upcoming task.
    
    ---
    
    ## 📅 Upcoming Deadlines
    
    | Deadline | Due Date & Time (Local) | Course / Project | Status |
    |----------|------------------------|------------------|--------|
    | **Project Proposal (Team X)** | **March 2

### Paste your traces from Opik here:
**THIS SCREENSHOT IS FOR DEMO ONLY!!!!** \
**MAKE SURE TO REPLACE IT BEFORE SUBMISSION** \
![image.png](attachment:image.png)